In [1]:
"""
Decentralized RS — Train/Test Split Ratio Experiment (ML-100K)
==============================================================
Compares three split strategies:  90/10  |  80/20  |  70/30
Val set is always 20% of the training portion (proportional).
Metrics tracked per ratio:
  • Test RMSE
  • Convergence speed (epochs to best val loss)
  • Communication cost (total commute × parameter bytes)

Drop in project root alongside src/ and dataset/.
Run:  python split_ratio_experiment.py
"""

from pathlib import Path
import os

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")



Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [2]:
import copy, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.optim import SGD
from tqdm import tqdm

from src.models.MatrixFactorization import UMF
from src.graphs import random_k_out_graph
from src.users import User
from src.data_utils import create_dataloader
from src.training.decentralized import (
    decentralized_train_n_epochs,
    decentralized_validate_loop,
)

In [3]:
para_vec= {
  "scalefree_userprop" : [0.006721468985407216, 0.3793755748581348, 0.7023494584199832],
  "scalefree_urs" : [0.00797255113179729, 0.7291631699209506, 0.7649689575684868],
  "scalefree_rs" : [0.043245636749499355, 0.24293301237845355, 0.6590721600407826],
  "scalefree_oaat" : [0.014505446034196021, 0.1281494707675557, 0.3063931184178566],
    
  "cycle_userprop" : [0.03448020025507248, 0.1530360406099725, 0.3265046312442892],
  "cycle_urs" : [0.015085184891905544, 0.32597756888723617, 0.9165691479123227],
  "cycle_rs" : [0.04518354114581989, 0.07432773840871296, 0.5104116722654509],
  "cycle_oaat" : [0.006051947990064438, 0.407449910177748, 0.6941867781038726],
    
  "random2_userprop" : [0.05973335259492166, 0.20270185084925238, 0.1],
  "random2_urs" : [0.03871364416669273, 0.14214480688557163, 0.4403378739685112],
  "random2_rs" : [0.03871364416669273, 0.14214480688557163, 0.01],
  "random2_oaat" : [0.012098247288774554, 0.051267232285266244, 0.5034632200402083],

  "random5_userprop" : [0.01214468819649195, 0.16071055871166323, 0.8930612583507401],
  "random5_urs" : [0.04664261576162963, 0.2261414992421005, 0.3645222958218734],
  "random5_rs" : [0.01864856189846265, 0.07043500222618476, 0.850748837624225],
  "random5_oaat" : [0.004358629931177893, 0.27784542450084454, 0.41295161556157467]
    
}

#lr = temp_para[0]
#weight_decay = temp_para[1]
#mom = temp_para[2]

In [4]:
warnings.filterwarnings("ignore")
torch.manual_seed(0)
np.random.seed(42)


# ──────────────────────────────────────────────────────────────────────────────
# Hyper-parameters  (mirrors your notebook exactly)
# ──────────────────────────────────────────────────────────────────────────────
HP = dict(
    n_factors    = 30,
    sparse       = False,
    batch_size   = 10,
    lr           = 0.004358629931177893,
    weight_decay = 0.27784542450084454,
    mom          = 0.41295161556157467,
    graph_seed   = 1,
    n_epochs     = 50,
    loader_type  = "oaat",
    # DP-SGD
    use_dp       = True,
    dp_clip_norm = 1.0,
    dp_noise_std = 0.01,
)

# Split ratios to benchmark: (train_frac, label)
SPLITS = [
    (0.90, "90/10"),
    (0.80, "80/20"),
    (0.70, "70/30"),
]

# Val is always 20 % of the training portion (proportional)
VAL_FRAC = 0.20


# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────
def make_train_test_split(full_df: pd.DataFrame, train_frac: float):
    """Split full interaction data into train / test by train_frac."""
    return train_test_split(full_df, train_size=train_frac, random_state=42)


def make_val_split(train_df: pd.DataFrame, val_frac: float = VAL_FRAC):
    """Carve val out of train proportionally."""
    return train_test_split(train_df, test_size=val_frac, random_state=0)


def build_users(n_users: int, n_items: int, hp: dict) -> dict:
    users = {}
    for i in tqdm(range(n_users), desc="  Init users", leave=False):
        model = UMF(n_items, n_factors=hp["n_factors"], sparse=hp["sparse"])
        opt   = SGD(model.parameters(), lr=hp["lr"],
                    weight_decay=hp["weight_decay"], momentum=hp["mom"])
        users[i] = User(id=i, model=model, optimizer=opt, model_name="umf")
    return users


def dp_epsilon(sigma, n_steps, n_train, batch_size, delta=1e-5):
    q = batch_size / n_train
    return np.sqrt(2 * n_steps * np.log(1 / delta)) * q / sigma


# ──────────────────────────────────────────────────────────────────────────────
# One experiment
# ──────────────────────────────────────────────────────────────────────────────
def run_experiment(label: str, train_df: pd.DataFrame,
                   val_df: pd.DataFrame, test_df: pd.DataFrame,
                   n_items: int, hp: dict) -> dict:

    print(f"\n{'─'*60}")
    print(f"  Ratio {label}  |  train={len(train_df)}  val={len(val_df)}"
          f"  test={len(test_df)}")
    print(f"{'─'*60}")

    n_users = train_df["user_id"].nunique()

    train_loader = create_dataloader(df=train_df, dl_type=hp["loader_type"],
                                     batch_size=hp["batch_size"])
    val_loader   = create_dataloader(df=val_df,  dl_type="rs")
    test_loader  = create_dataloader(df=test_df, dl_type="rs")

    users = build_users(n_users, n_items, hp)
    graph = random_k_out_graph(n=n_users, k=5, seed=hp["graph_seed"])

    torch.manual_seed(0)
    t0 = time.time()
    train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
        user_models=users,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=hp["n_epochs"],
        graph=graph,
    )
    elapsed = time.time() - t0

    test_rmse         = float(decentralized_validate_loop(users, test_loader))
    best_val          = float(min(val_losses))
    best_epoch        = int(np.argmin(val_losses)) + 1   # 1-indexed
    epochs_run        = len(train_losses)

    # Communication cost: commute × n_factors × 4 bytes (float32)
    param_bytes        = hp["n_factors"] * 4
    total_commute      = int(sum(commutes['commute']))
    comm_cost_mb       = round(total_commute * param_bytes / 1e6, 3)
    avg_commute_epoch  = round(total_commute / max(epochs_run, 1), 1)

    # Privacy budget at current noise level
    eps = dp_epsilon(hp["dp_noise_std"], epochs_run * len(train_loader),
                     len(train_df), hp["batch_size"])

    # ── NEW: per-epoch comm cost (bytes and MB) ──────────────────────────────
    # Commute count is constant per epoch (fixed graph), so cost per epoch
    # equals total_commute * param_bytes / epochs_run.
    comm_cost_per_epoch_mb  = round(total_commute * param_bytes / epochs_run / 1e6, 4)
    bytes_per_user_per_epoch = round(
        total_commute * param_bytes / epochs_run / n_users, 2
    )

    # ── NEW: cumulative comm cost (MB) at each epoch ──────────────────────────
    cumulative_comm_mb = [
        round(comm_cost_per_epoch_mb * (e + 1), 4)
        for e in range(epochs_run)
    ]

    # ── NEW: comm cost to reach RMSE threshold (using val loss as proxy) ──────
    RMSE_THRESHOLD = 0.93
    epochs_to_threshold = None
    cumul_at_threshold_mb = None
    for e, vl in enumerate(val_losses):
        if vl <= RMSE_THRESHOLD:
            epochs_to_threshold = e + 1          # 1-indexed
            cumul_at_threshold_mb = round(comm_cost_per_epoch_mb * (e + 1), 4)
            break

    result = dict(
        label                    = label,
        n_train                  = len(train_df),
        n_val                    = len(val_df),
        n_test                   = len(test_df),
        n_users                  = n_users,
        n_items                  = n_items,
        test_rmse                = round(test_rmse, 6),
        best_val_loss            = round(best_val, 6),
        best_epoch               = best_epoch,
        epochs_run               = epochs_run,
        train_losses             = [round(x, 6) for x in train_losses],
        val_losses               = [round(x, 6) for x in val_losses],
        time_per_epoch           = [round(x, 3) for x in time_per_epoch],
        commutes                 = commutes,
        total_commute            = total_commute,
        comm_cost_mb             = comm_cost_mb,
        avg_commute_epoch        = avg_commute_epoch,
        # ── NEW metrics ──────────────────────────────────────────────────────
        comm_cost_per_epoch_mb   = comm_cost_per_epoch_mb,
        bytes_per_user_per_epoch = bytes_per_user_per_epoch,
        cumulative_comm_mb       = cumulative_comm_mb,
        rmse_threshold           = RMSE_THRESHOLD,
        epochs_to_threshold      = epochs_to_threshold,
        cumul_at_threshold_mb    = cumul_at_threshold_mb,
        # ─────────────────────────────────────────────────────────────────────
        elapsed_s                = round(elapsed, 2),
        dp_epsilon               = round(eps, 4),
        dp_noise_std             = hp["dp_noise_std"],
    )

    print(f"  ✓  Test RMSE: {test_rmse:.4f}  |  Best Val @ epoch {best_epoch}"
          f"  |  Comm: {total_commute} MB  |  ε={eps:.2f}  |  {elapsed:.1f}s")
    return result



In [5]:
##%%
# ──────────────────────────────────────────────────────────────────────────────
# Data loading — ratings.csv pipeline
# ──────────────────────────────────────────────────────────────────────────────
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
#ratings = pd.read_csv("ratings.csv")
ratings = pd.read_csv('dataset/u.data',
                       sep='\t', names=column_names).iloc[:, 0:3]

# ── NEW: keep only users with at least 10 rated items ─────────────────────────
user_counts  = ratings['user_id'].value_counts()
active_users = user_counts[user_counts >= 10].index
ratings      = ratings[ratings['user_id'].isin(active_users)].reset_index(drop=True)
print(f"After ≥10-item filter: {len(ratings):,} ratings, {ratings['user_id'].nunique()} users retained")
# ───────────────────────────────────────────────────────────────────────────────

X = ratings[['user_id', 'item_id']].values
y = ratings['rating'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

X_train = pd.DataFrame(X_train, columns=['user_id', 'item_id'])
X_test  = pd.DataFrame(X_test,  columns=['user_id', 'item_id'])
y_train = pd.DataFrame(y_train, columns=['rating'])
y_test  = pd.DataFrame(y_test,  columns=['rating'])

# Only keep test users/items seen during training
frequent_users  = np.unique(X_train.user_id)
frequent_movies = np.unique(X_train.item_id)

drop_list = [
    i for i in range(X_test.shape[0])
    if (X_test.iloc[i].user_id  not in frequent_users) or
       (X_test.iloc[i].item_id not in frequent_movies)
]
X_test.drop(drop_list, inplace=True)
y_test.drop(drop_list, inplace=True)

# Re-index user/item IDs to contiguous integers
transaction = pd.concat([X_train, X_test])
customers   = np.unique(transaction.user_id.values)
items       = np.unique(transaction.item_id.values)

customer_id2index = {c: i for i, c in enumerate(customers)}
item_id2index     = {a: i for i, a in enumerate(items)}

X_train.user_id = X_train.user_id.map(customer_id2index)
X_train.item_id = X_train.item_id.map(item_id2index)
X_test.user_id  = X_test.user_id.map(customer_id2index)
X_test.item_id  = X_test.item_id.map(item_id2index)

# Merge features and labels back into single DataFrames
train_df = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)
test_df  = pd.concat([X_test,  y_test],  axis=1).reset_index(drop=True)

# Carve out validation set (20% of train, proportional)
train_df, val_df = train_test_split(train_df, test_size=VAL_FRAC, random_state=0)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

n_items = len(items)
print(f"Ratings: {len(ratings):,}  |  Users: {len(customers)}  |  Items: {n_items}")
print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

# ── Run experiments across split ratios ──────────────────────────────────────
# NOTE: train/test already split above (75/25). SPLITS here re-partition for
# systematic benchmarking; set SPLITS = [(1.0, '75/25')] to use the split above directly.
all_results = []
for train_frac, label in SPLITS:
    # Re-split from full merged data for each ratio
    full_df   = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
    tr, te    = train_test_split(full_df, train_size=train_frac, random_state=42)
    tr, va    = train_test_split(tr, test_size=VAL_FRAC, random_state=0)
    res = run_experiment(
        label,
        tr.reset_index(drop=True),
        va.reset_index(drop=True),
        te.reset_index(drop=True),
        n_items, HP
    )
    all_results.append(res)


After ≥10-item filter: 100,000 ratings, 943 users retained
Ratings: 100,000  |  Users: 943  |  Items: 1628
Train: 60,000  |  Val: 15,000  |  Test: 24,929

────────────────────────────────────────────────────────────
  Ratio 90/10  |  train=71948  val=17988  test=9993
────────────────────────────────────────────────────────────


  0%|          | 0/71948 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 3.7140 | Validation Loss: 1.6246 | Time Elapsed: 49.064565 sec |Commute: 359557 | Commute 
Cost: 3631071664

  0%|          | 0/71948 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 1.3466 | Validation Loss: 1.2651 | Time Elapsed: 48.997694 sec |Commute: 359557 | Commute 
Cost: 3631071664

...

  0%|          | 0/55960 [00:00<?, ?it/s]

Epoch 21 | Train Loss: 1.0848 | Validation Loss: 0.9527 | Time Elapsed: 36.188069 sec |Commute: 279644 | Commute 
Cost: 2824189280

Early stopping.

Total time elapsed: 781.0418461670633

  ✓  Test RMSE: 0.9513  |  Best Val @ epoch 20  |  Comm: 5872524 MB  |  ε=92.96  |  781.1s


In [6]:
# ── Summary Table 1: core metrics ────────────────────────────────────────
print("\n" + "═"*100)
print(f"{'Ratio':<8} {'TrainN':>7} {'TestN':>7} {'TestRMSE':>10}"
      f" {'BestEpoch':>10} {'TotalCommMB':>12} {'ε':>7}")
print("═"*100)
for r in all_results:
    print(f"{r['label']:<8} {r['n_train']:>7} {r['n_test']:>7}"
          f" {r['test_rmse']:>10.4f} {r['best_epoch']:>10}"
          f" {r['comm_cost_mb']:>12.2f} {r['dp_epsilon']:>7.2f}")
print("═"*100)

# ── Summary Table 2: new communication cost metrics ──────────────────────
print("\n── Communication cost breakdown ──")
print(f"{'Ratio':<8} {'CommPerEpochMB':>15} {'BytesPerUserPerEp':>18}"
      f" {'EpsToRMSE<0.93':>15} {'MBToRMSE<0.93':>14}")
print("─"*72)
for r in all_results:
    eps_str = str(r['epochs_to_threshold']) if r['epochs_to_threshold'] else "never"
    mb_str  = f"{r['cumul_at_threshold_mb']:.2f}" if r['cumul_at_threshold_mb'] else "never"
    print(f"{r['label']:<8} {r['comm_cost_per_epoch_mb']:>15.4f}"
          f" {r['bytes_per_user_per_epoch']:>18.2f}"
          f" {eps_str:>15} {mb_str:>14}")
print("─"*72)

# ── Summary Table 3: val loss per epoch (RMSE trajectory) ────────────────
print("\n── Validation loss (RMSE proxy) per epoch ──")
max_epochs = max(len(r['val_losses']) for r in all_results)
header = f"{'Epoch':>6}" + "".join(f"  {r['label']:>8}" for r in all_results)
print(header)
print("─" * len(header))
for e in range(max_epochs):
    row = f"{e+1:>6}"
    for r in all_results:
        vl = r['val_losses'][e] if e < len(r['val_losses']) else None
        row += f"  {vl:>8.4f}" if vl is not None else "         -"
    print(row)

# ── Summary Table 4: cumulative comm cost at each epoch ──────────────────
print("\n── Cumulative communication cost (MB) per epoch ──")
header2 = f"{'Epoch':>6}" + "".join(f"  {r['label']:>10}" for r in all_results)
print(header2)
print("─" * len(header2))
for e in range(max_epochs):
    row = f"{e+1:>6}"
    for r in all_results:
        cc = r['cumulative_comm_mb'][e] if e < len(r['cumulative_comm_mb']) else None
        row += f"  {cc:>10.2f}" if cc is not None else "           -"
    print(row)

 


════════════════════════════════════════════════════════════════════════════════════════════════════
Ratio     TrainN   TestN   TestRMSE  BestEpoch  TotalCommMB       ε
════════════════════════════════════════════════════════════════════════════════════════════════════
90/10      71948    9993     0.9368         20       906.08   81.98
80/20      63954   19986     0.9419         22       882.14   91.00
70/30      55960   29979     0.9513         20       704.70   92.96
════════════════════════════════════════════════════════════════════════════════════════════════════

── Communication cost breakdown ──
Ratio     CommPerEpochMB  BytesPerUserPerEp  EpsToRMSE<0.93  MBToRMSE<0.93
────────────────────────────────────────────────────────────────────────
90/10            43.1468           45754.87           never          never
80/20            38.3539           40672.24           never          never
70/30            33.5573           35585.66           never          never
───────────────